# _prev covariate 밀도 A/B 비교 (2026-09-02)

`blaine_prev` / `residue_prev` 를 어떻게 채우느냐를 2-way 로 비교한다.

| 분기 | 내용 |
|---|---|
| **sparse** | 현행 그대로 -- 실측 scheduled 행에만 값, 그 사이 hourly 행은 NaN |
| **ffill**  | 직전 실측값을 다음 측정 시점까지 forward-fill -- 모든 행이 "가장 최근 lab 값" 앵커를 명시적으로 보유 |

**고정 조건**: blaine `context_length=1024`, residue `context_length=1536`, 둘 다
`--finetune-mode full --prediction-length 4 --learning-rate 1e-6 --num-steps 1000 --batch-size 64`.
두 분기 모두 타깃값 / 분할 지점이 동일하고, backtest 평가 지점(실측 갱신 시점)의 `_prev` 값도 동일하다
-> 같은 평가 지점에서 비교 (naive / skill-score 분모 동일).

이 노트북은 GPU 를 쓰지 않는다 -- 저장된 `eval/backtest_*_prevdensity_*.csv` 만 읽는다.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

plt.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'config.py').exists())
sys.path.insert(0, str(ROOT))
import config as cfg
from eval.metrics_utils import compute_metrics, macro_average_mae, normalized_skill_score

TARGETS = cfg.TARGET_COLS
CONTEXT_LENGTHS = {'blaine': 1024, 'residue': 1536}

ARMS = ['sparse', 'ffill']
LABELS = {'sparse': 'sparse(현행)', 'ffill': 'ffill(dense)'}
COLORS = {'sparse': '#8b877a', 'ffill': '#2a78d6'}
FILES = {t: {arm: f'backtest_{t}_prevdensity_{arm}.csv' for arm in ARMS} for t in TARGETS}

print('targets:', TARGETS)
print('context_length:', CONTEXT_LENGTHS)

## 1. 결과 파일 로딩

In [ ]:
EVAL_DIR = ROOT / 'eval'

results = {}
missing = []
for target in TARGETS:
    results[target] = {}
    for arm in ARMS:
        p = EVAL_DIR / FILES[target][arm]
        if p.exists():
            results[target][arm] = pd.read_csv(p, parse_dates=['timestamp'])
        else:
            missing.append(str(p))

if missing:
    print('[missing]', len(missing), 'files:')
    for m in missing:
        print(' ', m)
else:
    print('전부 로딩 완료:')
    for target in TARGETS:
        ns = {arm: len(results[target][arm]) for arm in ARMS}
        print(f'  {target}: n = {ns}')
        if len(set(ns.values())) != 1:
            print('    \u26a0 두 분기의 n 이 다름 -- 평가 지점이 완전히 같지 않음')

## 2. 요약 지표 (pooled MAE/R²/spec + macro/정규화 skill-score)

In [ ]:
rows = []
for target in TARGETS:
    for arm in ARMS:
        df = results[target][arm]
        m = compute_metrics(df, cfg.SPEC_RANGES[target])
        rows.append({
            'target': target, 'arm': LABELS[arm], 'context_length': CONTEXT_LENGTHS[target],
            'n': len(df),
            'MAE': round(m['MAE'], 4), 'R2': round(m['R2'], 4),
            'spec_accuracy(%)': round(m['spec_accuracy'] * 100, 2),
            'interval_coverage(%)': round(m.get('interval_coverage', float('nan')) * 100, 2),
            'macro_MAE': round(macro_average_mae(df), 4),
            'skill_score': round(normalized_skill_score(df), 4),
        })

summary = pd.DataFrame(rows).set_index(['target', 'arm'])
summary

## 3. 비교 그래프 (sparse vs ffill)

타깃마다 네 지표(Skill-score / MAE / R² / Spec 정확도)를 작은 패널로 나눠 두 분기를 나란히 비교한다.
Skill-score / MAE 패널의 점선은 naive baseline 기준선이다.

In [ ]:
def plot_target(target):
    d = {arm: results[target][arm] for arm in ARMS}
    m = {arm: compute_metrics(d[arm], cfg.SPEC_RANGES[target]) for arm in ARMS}
    skill = {arm: normalized_skill_score(d[arm]) for arm in ARMS}
    naive_mae = float(np.mean([np.abs(d[a]['actual'] - d[a]['naive_pred']).mean() for a in ARMS]))

    fig, axes = plt.subplots(1, 4, figsize=(17, 4))
    fig.suptitle(f'{target}  (context_length={CONTEXT_LENGTHS[target]}, full fine-tuning, '
                 f'n={len(d[ARMS[0]])})', fontsize=13, fontweight='bold', y=1.05)

    def bars(ax, values, title, fmt='{:.4f}', naive=None, pct=False):
        xs = np.arange(len(ARMS))
        heights = [values[a] for a in ARMS]
        b = ax.bar(xs, heights, color=[COLORS[a] for a in ARMS], width=0.5)
        for rect, v in zip(b, heights):
            ax.text(rect.get_x() + rect.get_width() / 2, v, fmt.format(v) + ('%' if pct else ''),
                    ha='center', va='bottom', fontsize=9, fontweight='bold')
        if naive is not None:
            ax.axhline(naive, color='#c0392b', linestyle='--', linewidth=1.2)
            ax.text(len(ARMS) - 0.4, naive, f'naive {fmt.format(naive)}', fontsize=8,
                    color='#c0392b', va='bottom', ha='right')
        ax.set_xticks(xs)
        ax.set_xticklabels([LABELS[a] for a in ARMS], fontsize=9)
        ax.set_title(title, fontsize=10)
        lo, hi = ax.get_ylim()
        ax.set_ylim(lo - (hi - lo) * 0.05, hi + (hi - lo) * 0.18)

    bars(axes[0], skill, 'Skill-score (<1.0 = naive 이김)', fmt='{:.4f}', naive=1.0)
    bars(axes[1], {a: m[a]['MAE'] for a in ARMS}, 'MAE (낮을수록 좋음)', fmt='{:.3f}', naive=naive_mae)
    bars(axes[2], {a: m[a]['R2'] for a in ARMS}, 'R² (높을수록 좋음)', fmt='{:.4f}')
    bars(axes[3], {a: m[a]['spec_accuracy'] * 100 for a in ARMS}, 'Spec 정확도', fmt='{:.2f}', pct=True)
    plt.tight_layout()
    plt.show()


for t in TARGETS:
    plot_target(t)

## 4. 결론

_(실행 후 작성 -- `EXPERIMENT_LOG.md` 에도 기록)_

판단 기준:
- 1차 지표 = normalized skill-score. 개선폭이 재학습 노이즈(blaine ~0.004)보다 작으면 "차이 없음" -> 현행 sparse 유지.
- 유의미하면 seed 2~3개로 재확인 후 채택.
- ffill 이 유의미하게 나으면 정식 파이프라인 기본값을 `prev_density="ffill"` 로 전환 검토
  (build_dataset.py main(), dataset_utils, config 주석).